# Description

In this notebook, I will:
- Load the pre-trained model.
- Fine-tuning it on if-dataset

In [ ]:
import os, json, torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, List
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from transformers import (
    PreTrainedTokenizerFast,
    BertForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

In [ ]:
DATA_DIR = "if_dataset"               # where train.jsonl / validation.jsonl / test.jsonl live
MODEL_DIR = "mlm_model_bert"          # your pre-trained MLM model directory
TOKENIZER_JSON = "python_tokenizer.json"   # your trained tokenizer file
MAX_LEN = 512

In [ ]:
def load_jsonl(path: str) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

@dataclass
class IfMaskMLMBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

class IfMaskMLMDataset(Dataset):
    """
    Turns {"input": func_with_<mask>, "target": condition_text} into MLM examples:
      - Tokenize input
      - Tokenize target (without special tokens)
      - Replace single <mask> token with N copies of mask-token (N=len(target_ids))
      - Labels = -100 everywhere except the mask span, where labels=target_ids
    """
    def __init__(self, rows: List[Dict], tok: PreTrainedTokenizerFast, max_len: int = 512):
        self.rows = rows
        self.tok = tok
        self.max_len = max_len
        self.mask_id = tok.convert_tokens_to_ids(tok.mask_token)

        cleaned: List[IfMaskMLMBatch] = []
        for r in rows:
            inp = r["input"]
            tgt = r["target"]

            # Tokenize input and target
            enc_inp = tok(inp, add_special_tokens=True, truncation=False)
            enc_tgt = tok(tgt, add_special_tokens=False)

            # Find the single <mask> position in input ids
            input_ids = enc_inp["input_ids"]
            try:
                mask_pos = input_ids.index(self.mask_id)
            except ValueError:
                continue

            # Build expanded sequence: input_ids with <mask> replaced by len(target_ids) masks
            tgt_ids = enc_tgt["input_ids"]
            if len(tgt_ids) == 0:
                continue

            expanded = (
                input_ids[:mask_pos]
                + [self.mask_id] * len(tgt_ids)
                + input_ids[mask_pos + 1 :]
            )

            # Truncate if too long - simple policy: skip long example to keep code short
            if len(expanded) > max_len:
                continue

            # Labels: -100 everywhere, fill target ids at mask span
            labels = [-100] * len(expanded)
            for i, tid in enumerate(tgt_ids):
                labels[mask_pos + i] = tid

            # Attention mask
            attn = [1] * len(expanded)

            cleaned.append(
                IfMaskMLMBatch(
                    input_ids=torch.tensor(expanded, dtype=torch.long),
                    attention_mask=torch.tensor(attn, dtype=torch.long),
                    labels=torch.tensor(labels, dtype=torch.long),
                )
            )

        self.examples = cleaned

    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        ex = self.examples[i]
        return {
            "input_ids": ex.input_ids,
            "attention_mask": ex.attention_mask,
            "labels": ex.labels,
        }
        
        
def collate_mlm(batch, pad_id: int, pad_mult: int | None = 8):
    """
    Pads input_ids with pad_id, attention_mask with 0, labels with -100.
    Optionally pad to multiple of pad_mult (e.g., 8) for speed on GPU.
    """
    input_ids      = [ex["input_ids"] for ex in batch]
    attention_mask = [ex["attention_mask"] for ex in batch]
    labels         = [ex["labels"] for ex in batch]

    input_ids      = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels         = pad_sequence(labels, batch_first=True, padding_value=-100)

    if pad_mult:
        L = input_ids.size(1)
        rem = (-L) % pad_mult
        if rem:
            pad_cols = (0, rem)
            input_ids      = torch.nn.functional.pad(input_ids, pad_cols, value=pad_id)
            attention_mask = torch.nn.functional.pad(attention_mask, pad_cols, value=0)
            labels         = torch.nn.functional.pad(labels, pad_cols, value=-100)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
def mlm_accuracy(eval_pred):
    logits, labels = eval_pred  # logits: (N, T, V), labels: (N, T)
    preds = logits.argmax(-1)
    mask = labels != -100
    total = mask.sum()
    if total == 0:
        return {"mlm_acc": 0.0}
    correct = (preds[mask] == labels[mask]).sum()
    return {"mlm_acc": (correct / total).item()}

In [ ]:
# Tokenizer
tok = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER_JSON)
tok.add_special_tokens({
    "pad_token": "<pad>", "unk_token": "<unk>", "mask_token": "<mask>",
    "bos_token": "<s>", "eos_token": "</s>",
})

In [ ]:
train_rows = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
val_rows   = load_jsonl(os.path.join(DATA_DIR, "validation.jsonl"))
test_rows  = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))

train_ds = IfMaskMLMDataset(train_rows, tok, MAX_LEN)
val_ds   = IfMaskMLMDataset(val_rows, tok, MAX_LEN)
test_ds  = IfMaskMLMDataset(test_rows, tok, MAX_LEN)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
model = BertForMaskedLM.from_pretrained(MODEL_DIR)

data_collator = lambda batch: collate_mlm(batch, pad_id=tok.pad_token_id, pad_mult=8)

args = TrainingArguments(
    output_dir="if_mlm_finetuned",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-4,
    num_train_epochs=3,
    logging_steps=50,
    do_eval=True,
    save_steps=1000,
    fp16=torch.cuda.is_available(),
    report_to="none",   # remove if your transformers is too old
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=data_collator,
    compute_metrics=mlm_accuracy,
)

trainer.train()

trainer.save_model("if_mlm_finetuned")
tok.save_pretrained("if_mlm_finetuned")
print("[DONE] Saved fine-tuned model to if_mlm_finetuned")

In [ ]:
# Evaluation
with torch.no_grad():
    list_test_acc = []
    for test_batch in test_ds:
        test_batch = {k: v.unsqueeze(0).to(trainer.args.device) for k, v in test_batch.items()}
        outputs = model(**test_batch)
        
        logits = outputs.logits
        preds = logits.argmax(-1)
    
        acc = mlm_accuracy((logits.cpu().numpy(), test_batch['labels'].cpu().numpy()))
        list_test_acc.append(acc['mlm_acc'])
        
    mean_test_acc = np.mean(list_test_acc)
    print(f"Test MLM Accuracy: {mean_test_acc:.4f}")

In [ ]:
@torch.no_grad()
def predict_if_condition(func_with_mask: str,
                         tok,
                         model,
                         try_lengths=range(2, 13),   # candidate token lengths to try
                         max_len=512,
                         device=None):
    """
    Given a function string where a single <mask> token replaces the condition,
    predict the most likely condition text using a fine-tuned BERT-MLM model.
    Tries multiple lengths and returns the best-scoring decoded text.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Load model & tokenizer
    mask_id = tok.convert_tokens_to_ids(tok.mask_token)
    model.eval()

    # Tokenize input once
    enc = tok(func_with_mask, add_special_tokens=True, truncation=True, max_length=max_len, return_tensors="pt")
    input_ids = enc["input_ids"][0].tolist()
    attn      = enc["attention_mask"][0].tolist()

    # Find the single <mask> position
    if mask_id not in input_ids:
        raise ValueError("No <mask> token found in input.")
    mask_pos = input_ids.index(mask_id)

    best_text = ""
    best_score = float("-inf")

    for L in try_lengths:
        # Build expanded sequence: replace the single mask with L masks
        expanded_ids = input_ids[:mask_pos] + [mask_id]*L + input_ids[mask_pos+1:]
        expanded_attn = attn[:mask_pos] + [1]*L + attn[mask_pos+1:]

        # Truncate safely (skip if masked span would be chopped)
        if len(expanded_ids) > max_len:
            continue

        # Run model
        tens_ids  = torch.tensor([expanded_ids], dtype=torch.long, device=device)
        tens_attn = torch.tensor([expanded_attn], dtype=torch.long, device=device)
        out = model(input_ids=tens_ids, attention_mask=tens_attn)
        logits = out.logits[0]  # (T, V)

        # Take predictions for the L mask slots
        span_logits = logits[mask_pos:mask_pos+L]        # (L, V)
        probs = F.log_softmax(span_logits, dim=-1)       # log-probs
        pred_ids = probs.argmax(dim=-1)                  # (L,)
        score = probs.gather(1, pred_ids.unsqueeze(1)).sum().item()

        # Decode predicted token ids into string
        pred_text = tok.decode(pred_ids.tolist(), skip_special_tokens=True).strip()

        if score > best_score and pred_text:
            best_score = score
            best_text = pred_text

    return best_text, best_score

In [ ]:
example = """\
def is_even(x):
    if x % 2 == <mask>:
        return "even"
    else:
        return "odd"
"""

pred, score = predict_if_condition(example, tok, model,
                                  try_lengths=range(2, 13),
                                  max_len=512)  
print("Predicted condition:", pred)
print("Score:", score)